
# Predicción del Precio de Autos - Rusty Bargain

Rusty Bargain es un servicio de coches usados que desea crear una app para estimar el valor de mercado de un vehículo.

## 🎯 Objetivos del proyecto

- Predecir el precio de un coche en euros.
- Comparar modelos diferentes: regresión lineal, árbol de decisión, bosque aleatorio y potenciación del gradiente (LightGBM).
- Implementar manualmente una regresión lineal con descenso por gradiente.
- Medir:
  - Precisión: RECM (RMSE)
  - Tiempo de entrenamiento
  - Velocidad de predicción

## Consideraciones

- La regresión lineal sirve como prueba de cordura.
- Potenciación del gradiente debe funcionar mejor que regresión lineal, si no, algo está mal.
- LightGBM y CatBoost manejan categóricas; XGBoost requiere OHE.
- Usa `%%time` para medir tiempos en Jupyter.


## Carga de datos y librerías

In [4]:
#conda install -c conda-forge lightgbm

In [6]:
#!pip install lightgbm

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

In [3]:
df = pd.read_csv('car_data.csv')
df.head()

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,24/03/2016 11:52,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,24/03/2016 00:00,0,70435,07/04/2016 03:16
1,24/03/2016 10:58,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,14/03/2016 00:00,0,90480,05/04/2016 12:47
3,17/03/2016 16:54,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,17/03/2016 00:00,0,91074,17/03/2016 17:40
4,31/03/2016 17:25,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,31/03/2016 00:00,0,60437,06/04/2016 10:17


## Análisis exploratorio y limpieza

En esta celda realizamos una serie de pasos clave para preparar los datos antes de entrenar nuestros modelos:

1. **Filtrado de valores extremos (outliers):**
   - `Price` se restringe entre 100 y 50,000 euros para eliminar errores o valores atípicos.
   - `RegistrationYear` se limita entre 1950 y 2022 para asegurarse de que los años sean válidos.
   - `Power` (potencia del vehículo) se restringe entre 10 y 500 caballos de fuerza para evitar valores irreales.

2. **Eliminación de columnas irrelevantes:**
   - Se eliminan columnas que no aportan valor predictivo como:
     - `NumberOfPictures` (todos los valores son 0)
     - Fechas (`DateCrawled`, `DateCreated`, `LastSeen`)
     - `PostalCode` (identificador geográfico muy granular)

3. **Eliminación de filas con valores faltantes:**
   - `df.dropna()` descarta cualquier fila que contenga `NaN`, garantizando que el modelo no se entrene con datos incompletos.

4. **Codificación de variables categóricas:**
   - Se seleccionan las columnas categóricas y se transforman en variables dummy con `pd.get_dummies()`, usando `drop_first=True` para evitar multicolinealidad.

5. **Definición de variables predictoras y objetivo:**
   - `X`: todas las columnas excepto `'Price'`, que serán las características usadas para predecir.
   - `y`: la columna `'Price'`, que es nuestro objetivo.

6. **División del conjunto de datos:**
   - Se divide en conjunto de entrenamiento (`X_train`, `y_train`) y prueba (`X_test`, `y_test`) usando un 75% para entrenar y 25% para evaluar.


In [5]:
df = df[df['Price'].between(100, 50000)]
df = df[df['RegistrationYear'].between(1950, 2022)]
df = df[df['Power'].between(10, 500)]

In [6]:
df = df.drop(columns=['NumberOfPictures', 'DateCrawled', 'DateCreated', 'LastSeen', 'PostalCode'])
df = df.dropna()

In [7]:
categorical = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']
df = pd.get_dummies(df, columns=categorical, drop_first=True)

In [8]:
df.head()

,Price,RegistrationYear,Power,Mileage,RegistrationMonth,VehicleType_convertible,VehicleType_coupe,VehicleType_other,VehicleType_sedan,VehicleType_small,...,Brand_seat,Brand_skoda,Brand_smart,Brand_subaru,Brand_suzuki,Brand_toyota,Brand_trabant,Brand_volkswagen,Brand_volvo,NotRepaired_yes
3,1500,2001,75,150000,6,False,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
4,3600,2008,69,90000,7,False,False,False,False,True,...,False,True,False,False,False,False,False,False,False,False
5,650,1995,102,150000,10,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,True
6,2200,2004,109,150000,8,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
10,2000,2004,105,150000,12,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False


In [9]:
X = df.drop('Price', axis=1)
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## Modelo 1: Regresión Lineal

En esta celda entrenamos un modelo de regresión lineal utilizando `scikit-learn`. Este modelo sirve como una **prueba de cordura** (baseline), ya que su simplicidad nos permite comparar si modelos más complejos realmente mejoran el desempeño.

### Explicación del código:

- `%%time`: es un comando mágico de Jupyter que mide el **tiempo total de ejecución** de la celda, útil para comparar el costo computacional entre modelos.
- `LinearRegression()`: crea una instancia del modelo de regresión lineal.
- `lr.fit(X_train, y_train)`: entrena el modelo con los datos de entrenamiento.
- `lr.predict(X_test)`: realiza predicciones sobre el conjunto de prueba.
- `mean_squared_error(..., squared=False)`: calcula la **Raíz del Error Cuadrático Medio (RMSE)**, una métrica que indica qué tan lejos, en promedio, están las predicciones de los valores reales (en euros).
- `print(...)`: muestra el valor de RMSE, que nos servirá para comparar con otros modelos.

Un buen modelo debería obtener una RMSE menor que el de esta regresión lineal. Si no es así, significa que el modelo complejo no está aprendiendo correctamente.


In [10]:
%%time

lr = LinearRegression()
lr.fit(X_train, y_train)
preds_lr = lr.predict(X_test)
rmse_lr = mean_squared_error(y_test, preds_lr, squared=False)
print(f"RMSE Linear Regression: {rmse_lr:.2f}")

RMSE Linear Regression: 2502.57
CPU times: total: 4.34 s
Wall time: 4.39 s


C:\Users\ljpca\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


## Modelo 2: Árbol de Decisión

En esta celda entrenamos un modelo de árbol de decisión, un algoritmo basado en reglas que divide los datos en segmentos según condiciones lógicas para hacer predicciones.

### Explicación del código:

- `%%time`: mide el tiempo total de ejecución de la celda, lo que nos permite comparar su velocidad con otros modelos.
- `DecisionTreeRegressor(max_depth=10, random_state=42)`: crea un árbol de regresión limitado a una profundidad máxima de 10 niveles para evitar sobreajuste. El `random_state` asegura reproducibilidad.
- `tree.fit(X_train, y_train)`: entrena el árbol con los datos de entrenamiento.
- `tree.predict(X_test)`: genera predicciones sobre los datos de prueba.
- `mean_squared_error(..., squared=False)`: calcula el **RMSE**, que nos da una idea de qué tan buena es la predicción en unidades monetarias (euros).
- `print(...)`: muestra el valor del RMSE obtenido por el árbol.

In [11]:

%%time
tree = DecisionTreeRegressor(max_depth=10, random_state=42)
tree.fit(X_train, y_train)
preds_tree = tree.predict(X_test)
rmse_tree = mean_squared_error(y_test, preds_tree, squared=False)
print(f"RMSE Decision Tree: {rmse_tree:.2f}")


RMSE Decision Tree: 1948.47
CPU times: total: 1.78 s
Wall time: 3.45 s


C:\Users\ljpca\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

# Creamos modelo vacio
tree = DecisionTreeRegressor(random_state=42)

# definir cuadricula
param_grid = {
    'max_depth': [10, 100, 150],
    'min_samples_split': [2, 5]
    #,'min_samples_leaf': [1, 2]
}

rmse_scorer = make_scorer(mean_squared_error, greater_is_better=False, squared=False)

grid_search = GridSearchCV(estimator=tree, param_grid=param_grid,
                           scoring=rmse_scorer, cv=2, n_jobs=-1)

#%%time
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

preds_tree = best_model.predict(X_test)
rmse_tree = mean_squared_error(y_test, preds_tree, squared=False)
print(f"Mejor combinación: {grid_search.best_params_}")
print(f"RMSE del mejor modelo: {rmse_tree:.2f}")

Mejor combinación: {'max_depth': 10, 'min_samples_split': 5}
RMSE del mejor modelo: 1948.69


C:\Users\ljpca\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


## Modelo 3: Bosque Aleatorio

En esta celda entrenamos un modelo de **bosque aleatorio**, que consiste en un conjunto de múltiples árboles de decisión entrenados sobre subconjuntos aleatorios de datos y características.

### Explicación del código:

- `%%time`: registra el tiempo total que tarda la celda en ejecutarse, útil para comparar el rendimiento computacional entre modelos.
- `RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)`: se crea un bosque aleatorio compuesto por 100 árboles (`n_estimators`). Cada árbol tiene una profundidad máxima de 15 (`max_depth`) para evitar sobreajuste.
- `forest.fit(X_train, y_train)`: entrena el conjunto de árboles sobre los datos de entrenamiento.
- `forest.predict(X_test)`: predice los precios utilizando el promedio de las predicciones de todos los árboles.
- `mean_squared_error(..., squared=False)`: calcula el **RMSE**, que mide la calidad de las predicciones en euros.
- `print(...)`: muestra el RMSE para este modelo.

In [20]:

%%time
forest = RandomForestRegressor(n_estimators=10, max_depth=3, random_state=42)
forest.fit(X_train, y_train)
preds_forest = forest.predict(X_test)
rmse_forest = mean_squared_error(y_test, preds_forest, squared=False)
print(f"RMSE Random Forest: {rmse_forest:.2f}")


RMSE Random Forest: 2772.93
CPU times: total: 3.58 s
Wall time: 6.22 s


C:\Users\ljpca\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


## ⚡ Modelo 4: LightGBM

En esta celda entrenamos un modelo de **potenciación del gradiente** utilizando la librería **LightGBM**, optimizada para ser muy rápida y eficiente incluso en grandes volúmenes de datos.

### Explicación del código:

- `%%time`: mide el tiempo de ejecución completo de la celda para evaluar su velocidad.
- `lgb.Dataset(X_train, y_train)`: convierte los datos de entrenamiento en el formato específico que espera LightGBM.
- `params = {...}`: se definen los hiperparámetros del modelo:
  - `'objective': 'regression'` indica que es un problema de regresión.
  - `'metric': 'rmse'` especifica que la métrica de evaluación será RMSE.
  - `'learning_rate': 0.1` controla la tasa con la que se actualizan los árboles.
  - `'max_depth': 10` limita la profundidad de cada árbol.
  - `'verbose': -1` evita que LightGBM imprima mensajes durante el entrenamiento.
- `lgb.train(...)`: entrena el modelo con los parámetros definidos y 100 árboles (`num_boost_round=100`).
- `gbm.predict(X_test)`: realiza predicciones sobre los datos de prueba.
- `mean_squared_error(..., squared=False)`: calcula el **RMSE** para evaluar el error de las predicciones.
- `print(...)`: muestra el RMSE final del modelo LightGBM.

LightGBM es una implementación avanzada de **Gradient Boosting** que tiende a ser más rápida y eficiente que métodos tradicionales. Además, puede manejar directamente variables categóricas (aunque aquí ya fueron codificadas con one-hot).

In [21]:

%%time
lgb_train = lgb.Dataset(X_train, y_train)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'max_depth': 10,
    'verbose': -1
}

gbm = lgb.train(params, lgb_train, num_boost_round=100)
preds_lgb = gbm.predict(X_test)

rmse_lgb = mean_squared_error(y_test, preds_lgb, squared=False)

print(f"RMSE LightGBM: {rmse_lgb:.2f}")


RMSE LightGBM: 1650.43
CPU times: total: 7.36 s
Wall time: 1.77 s


C:\Users\ljpca\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


## Modelo 5: Regresión Lineal con Descenso por Gradiente (manual)

En esta celda implementamos desde cero una regresión lineal entrenada mediante el algoritmo de **descenso por gradiente**. Esto nos ayuda a entender cómo se optimizan los coeficientes del modelo sin depender de bibliotecas externas.

### Explicación del código:

#### Normalización:
- `StandardScaler()` estandariza las características para que tengan media 0 y desviación estándar 1.
- Esto es importante porque el descenso por gradiente es sensible a la escala de los datos.

#### Inicialización:
- `theta = np.zeros(...)`: inicializa los pesos del modelo en ceros.
- `lr = 0.01`: define la tasa de aprendizaje (learning rate).
- `n_iter = 1000`: número de iteraciones del algoritmo.

#### Descenso por Gradiente:
- En cada iteración se calculan los **gradientes** del error cuadrático medio respecto a los pesos.
- Luego se actualizan los pesos `theta` en la dirección opuesta al gradiente (minimizando el error).

#### Predicción y Evaluación:
- `y_pred_gd = X_test_s.dot(theta)`: realiza la predicción como producto escalar entre pesos y características.
- `mean_squared_error(..., squared=False)`: calcula el **RMSE** para medir la calidad del modelo.
- `print(...)`: muestra el RMSE final.

Esta implementación es útil como ejercicio académico. Aunque no es tan eficiente como `scikit-learn`, demuestra cómo funcionan internamente los métodos de optimización de modelos lineales.

In [22]:
%%time

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

# Descenso por gradiente
theta = np.zeros(X_train_s.shape[1])
lr = 0.01
n_iter = 100

for i in range(n_iter):
    gradients = -2 / len(X_train_s) * X_train_s.T.dot(y_train_s - X_train_s.dot(theta))
    theta -= lr * gradients

y_pred_gd = X_test_s.dot(theta)
rmse_gd = mean_squared_error(y_test_s, y_pred_gd, squared=False)
print(f"RMSE Gradiente Descendente: {rmse_gd:.2f}")


RMSE Gradiente Descendente: 5887.34


C:\Users\ljpca\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


## 📊 Comparación final de modelos

Resumiendo:

In [23]:

modelos = ['LinearRegression', 'DecisionTree', 'RandomForest', 'LightGBM', 'GradienteDesc']
rmses = [rmse_lr, rmse_tree, rmse_forest, rmse_lgb, rmse_gd]
pd.DataFrame({'Modelo': modelos, 'RMSE': rmses}).sort_values(by='RMSE')


,Modelo,RMSE
2,RandomForest,1626.397687
3,LightGBM,1650.427274
1,DecisionTree,1948.469515
0,LinearRegression,2502.574492
4,GradienteDesc,5887.340950
